# 🧠 Detección de Enfermedades Retinianas con Deep Learning


**Problema:** Clasificación de retinopatía diabética en imágenes del fondo de ojo  
**Dataset:** APTOS 2019 Blindness Detection (Kaggle)  
**Objetivo:** Clasificar la severidad de retinopatía en 5 niveles (0=No DR, 4=Proliferative DR)

---
### 📋 Estado del Pipeline

| Etapa | Estado |
|---|---|
| ✅ 1. Data Retrieval | Completo |
| ✅ 2. Data Processing & Wrangling | Completo |
| ✅ 3. Feature Extraction & Engineering | Completo |
| ✅ 4. Feature Scaling & Selection | Completo |
| ⏳ 5. Modeling (ML Algorithm) | **PENDIENTE** |
| ⏳ 6. Model Evaluation & Tuning | **PENDIENTE** |
| ⏳ 7. Deployment & Monitoring | **PENDIENTE** |

---

## 📦 0. Instalación de Dependencias

In [ ]:
# Instalar librerías necesarias
!pip install -q kaggle efficientnet tensorflow-addons opencv-python-headless grad-cam
!pip install -q imbalanced-learn seaborn scikit-learn matplotlib pandas numpy

In [ ]:
# Importaciones generales
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import warnings
warnings.filterwarnings('ignore')

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Machine Learning clásico
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE

# Verificar GPU
print('TensorFlow version:', tf.__version__)
print('GPU disponible:', tf.config.list_physical_devices('GPU'))
print('Dispositivo activo:', tf.test.gpu_device_name() or 'CPU')

---
## 🗄️ 1. DATA RETRIEVAL
> **Objetivo:** Obtener el dataset APTOS 2019 desde Kaggle y verificar su integridad.

In [ ]:
# ──────────────────────────────────────────────
# 1.1 Configuración de Kaggle API
# ──────────────────────────────────────────────
# Sube tu archivo kaggle.json (obtenido de https://www.kaggle.com/account)
from google.colab import files

print('Sube tu archivo kaggle.json:')
uploaded = files.upload()

# Configurar credenciales
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print('✅ Kaggle configurado correctamente')

In [ ]:
# ──────────────────────────────────────────────
# 1.2 Descarga del Dataset
# ──────────────────────────────────────────────
!kaggle competitions download -c aptos2019-blindness-detection -p /content/data
!unzip -q /content/data/aptos2019-blindness-detection.zip -d /content/data/
print('✅ Dataset descargado y descomprimido')

In [ ]:
# ──────────────────────────────────────────────
# 1.3 Carga y verificación inicial
# ──────────────────────────────────────────────
BASE_DIR = '/content/data'
TRAIN_IMG_DIR = os.path.join(BASE_DIR, 'train_images')
TEST_IMG_DIR  = os.path.join(BASE_DIR, 'test_images')

# Cargar CSV principal
df_train = pd.read_csv(os.path.join(BASE_DIR, 'train.csv'))
df_test  = pd.read_csv(os.path.join(BASE_DIR, 'test.csv'))

print('=' * 50)
print(f'Registros de entrenamiento : {len(df_train)}')
print(f'Registros de test          : {len(df_test)}')
print(f'Imágenes en train_images/  : {len(os.listdir(TRAIN_IMG_DIR))}')
print('=' * 50)
display(df_train.head(10))

# Mapeo de etiquetas
LABEL_MAP = {
    0: 'No DR',
    1: 'Mild',
    2: 'Moderate',
    3: 'Severe',
    4: 'Proliferative DR'
}
df_train['diagnosis_label'] = df_train['diagnosis'].map(LABEL_MAP)
print('\n✅ Dataset cargado correctamente')

---
## 🧹 2. DATA PROCESSING & WRANGLING
> **Objetivo:** Limpiar los datos, analizar distribuciones, tratar imbalance de clases y preparar splits.

In [ ]:
# ──────────────────────────────────────────────
# 2.1 Análisis Exploratorio de Datos (EDA)
# ──────────────────────────────────────────────
print('--- Información general ---')
print(df_train.info())
print('\n--- Valores nulos ---')
print(df_train.isnull().sum())
print('\n--- Estadísticas descriptivas ---')
display(df_train.describe())

In [ ]:
# ──────────────────────────────────────────────
# 2.2 Visualización de distribución de clases
# ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Conteo por clase
class_counts = df_train['diagnosis_label'].value_counts()
colors = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c', '#8e44ad']

axes[0].bar(class_counts.index, class_counts.values, color=colors)
axes[0].set_title('Distribución de Clases (Retinopatía Diabética)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Nivel de Severidad')
axes[0].set_ylabel('Cantidad de imágenes')
axes[0].tick_params(axis='x', rotation=15)
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(class_counts.values, labels=class_counts.index,
            autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Proporción de Clases', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('/content/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('⚠️  Se observa fuerte desbalance de clases — se aplicará Class Weighting + Augmentation')

In [ ]:
# ──────────────────────────────────────────────
# 2.3 Visualización de imágenes por clase
# ──────────────────────────────────────────────
def load_image(img_id, directory, size=(224, 224)):
    path = os.path.join(directory, img_id + '.png')
    img  = cv2.imread(path)
    img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img  = cv2.resize(img, size)
    return img

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for label_id, (label_name, ax) in enumerate(zip(LABEL_MAP.values(), axes)):
    sample = df_train[df_train['diagnosis'] == label_id].iloc[0]
    img = load_image(sample['id_code'], TRAIN_IMG_DIR)
    ax.imshow(img)
    ax.set_title(f'Clase {label_id}\n{label_name}', fontsize=10, fontweight='bold')
    ax.axis('off')

plt.suptitle('Ejemplos de Imágenes por Nivel de Severidad', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/sample_images_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ──────────────────────────────────────────────
# 2.4 Preprocesamiento de imágenes médicas
# (Ben Graham preprocessing — técnica estándar para imágenes retinianas)
# ──────────────────────────────────────────────
def preprocess_retinal_image(img_id, directory, size=380):
    """
    Aplica el preprocesamiento Ben Graham:
    - Recorte circular del fondo de ojo
    - Sustracción de Gaussian blur para resaltar estructuras
    - Normalización al rango [0, 1]
    """
    path = os.path.join(directory, img_id + '.png')
    img  = cv2.imread(path)
    img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img  = cv2.resize(img, (size, size))

    # Ben Graham: realce de contraste
    img_blur = cv2.GaussianBlur(img, (0, 0), sigmaX=size // 30)
    img_enhanced = cv2.addWeighted(img, 4, img_blur, -4, 128)

    # Máscara circular
    mask = np.zeros(img_enhanced.shape, dtype=np.uint8)
    center = (size // 2, size // 2)
    radius = int(size * 0.45)
    cv2.circle(mask, center, radius, (1, 1, 1), -1)
    img_final = img_enhanced * mask + 128 * (1 - mask)

    return img_final.astype(np.float32) / 255.0

# Comparación visual: Original vs Preprocesada
sample_id = df_train.iloc[0]['id_code']
orig = load_image(sample_id, TRAIN_IMG_DIR, (380, 380))
proc = preprocess_retinal_image(sample_id, TRAIN_IMG_DIR)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.imshow(orig / 255.0); ax1.set_title('Original', fontweight='bold'); ax1.axis('off')
ax2.imshow(proc);         ax2.set_title('Ben Graham Preprocessed', fontweight='bold'); ax2.axis('off')
plt.suptitle('Preprocesamiento de Imagen Retiniana', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/preprocessing_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Función de preprocesamiento Ben Graham aplicada correctamente')

In [ ]:
# ──────────────────────────────────────────────
# 2.5 Train / Validation Split (Estratificado)
# ──────────────────────────────────────────────
df_train['image_path'] = df_train['id_code'].apply(
    lambda x: os.path.join(TRAIN_IMG_DIR, x + '.png')
)

# Split estratificado: 80% train / 20% validation
train_df, val_df = train_test_split(
    df_train,
    test_size=0.20,
    random_state=42,
    stratify=df_train['diagnosis']
)

print('=' * 45)
print(f'  Train set : {len(train_df):>5} imágenes')
print(f'  Val   set : {len(val_df):>5} imágenes')
print('=' * 45)
print('\nDistribución de clases en Train:')
print(train_df['diagnosis_label'].value_counts())
print('\n✅ Split estratificado completado')

---
## ⚙️ 3. FEATURE EXTRACTION & ENGINEERING
> **Objetivo:** Construir el pipeline de extracción de características usando Transfer Learning con EfficientNetB4 y Data Augmentation avanzado.

In [ ]:
# ──────────────────────────────────────────────
# 3.1 Data Augmentation (nivel médico)
# ──────────────────────────────────────────────
IMG_SIZE   = 380
BATCH_SIZE = 16
NUM_CLASSES = 5

# Augmentation para entrenamiento (conservador para imágenes médicas)
train_datagen = ImageDataGenerator(
    preprocessing_function=lambda x: x / 255.0,
    rotation_range=360,        # Rotación completa (retina es circular)
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    brightness_range=[0.85, 1.15],
    fill_mode='constant',
    cval=0
)

# Sin augmentation para validación
val_datagen = ImageDataGenerator(
    preprocessing_function=lambda x: x / 255.0
)

# Convertir etiquetas a string para el generador
train_df['diagnosis_str'] = train_df['diagnosis'].astype(str)
val_df['diagnosis_str']   = val_df['diagnosis'].astype(str)

train_generator = train_datagen.flow_from_dataframe(
    dataframe    = train_df,
    x_col        = 'image_path',
    y_col        = 'diagnosis_str',
    target_size  = (IMG_SIZE, IMG_SIZE),
    batch_size   = BATCH_SIZE,
    class_mode   = 'sparse',
    shuffle      = True,
    seed         = 42
)

val_generator = val_datagen.flow_from_dataframe(
    dataframe    = val_df,
    x_col        = 'image_path',
    y_col        = 'diagnosis_str',
    target_size  = (IMG_SIZE, IMG_SIZE),
    batch_size   = BATCH_SIZE,
    class_mode   = 'sparse',
    shuffle      = False
)

print('✅ Generadores de datos configurados')
print(f'   Pasos por epoch (train): {len(train_generator)}')
print(f'   Pasos por epoch (val)  : {len(val_generator)}')

In [ ]:
# ──────────────────────────────────────────────
# 3.2 Visualización del Data Augmentation
# ──────────────────────────────────────────────
aug_gen = ImageDataGenerator(
    rotation_range=360,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.15,
    brightness_range=[0.8, 1.2],
    fill_mode='constant', cval=0
)

sample_img = load_image(train_df.iloc[0]['id_code'], TRAIN_IMG_DIR, (380, 380))
sample_img = sample_img.reshape((1,) + sample_img.shape)

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()
axes[0].imshow(sample_img[0] / 255.0)
axes[0].set_title('Original', fontweight='bold', color='green')
axes[0].axis('off')

for i, batch in enumerate(aug_gen.flow(sample_img, batch_size=1, seed=i), start=1):
    if i >= 10: break
    axes[i].imshow(batch[0] / 255.0)
    axes[i].set_title(f'Augmented #{i}', fontweight='bold')
    axes[i].axis('off')

plt.suptitle('Data Augmentation — Variaciones generadas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/data_augmentation_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Data Augmentation visualizado')

In [ ]:
# ──────────────────────────────────────────────
# 3.3 Arquitectura: Transfer Learning con EfficientNetB4
# ──────────────────────────────────────────────
def build_efficientnet_model(num_classes=5, img_size=380, learning_rate=1e-4):
    """
    Construye el modelo basado en EfficientNetB4 con capas personalizadas.
    Estrategia: Feature extraction primero, luego fine-tuning.
    """
    # Base preentrenada en ImageNet
    base_model = EfficientNetB4(
        weights     = 'imagenet',
        include_top = False,
        input_shape = (img_size, img_size, 3)
    )
    base_model.trainable = False  # Congelar en fase 1

    # Cabeza de clasificación personalizada
    inputs = keras.Input(shape=(img_size, img_size, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation='relu', name='fc1')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu', name='fc2')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='output')(x)

    model = keras.Model(inputs, outputs, name='EfficientNetB4_RetinalDR')

    model.compile(
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate),
        loss      = 'sparse_categorical_crossentropy',
        metrics   = ['accuracy',
                     keras.metrics.SparseTopKCategoricalAccuracy(k=2, name='top2_acc')]
    )
    return model, base_model

model, base_model = build_efficientnet_model(num_classes=NUM_CLASSES, img_size=IMG_SIZE)
model.summary()
print(f'\n✅ Modelo construido: {model.count_params():,} parámetros totales')
print(f'   Parámetros entrenables: {sum(tf.size(v).numpy() for v in model.trainable_variables):,}')

---
## 📊 4. FEATURE SCALING & SELECTION
> **Objetivo:** Manejar el desbalance de clases, calcular class weights, y configurar callbacks de regularización.

In [ ]:
# ──────────────────────────────────────────────
# 4.1 Manejo de Desbalance de Clases (Class Weighting)
# ──────────────────────────────────────────────
class_weights_array = compute_class_weight(
    class_weight = 'balanced',
    classes      = np.arange(NUM_CLASSES),
    y            = train_df['diagnosis'].values
)
class_weights = {i: w for i, w in enumerate(class_weights_array)}

print('Class Weights calculados (para compensar desbalance):')
print('─' * 40)
for cls_id, weight in class_weights.items():
    print(f'  Clase {cls_id} ({LABEL_MAP[cls_id]:<18}): {weight:.4f}')

# Visualización
fig, ax = plt.subplots(figsize=(8, 4))
classes = [LABEL_MAP[i] for i in range(NUM_CLASSES)]
weights = [class_weights[i] for i in range(NUM_CLASSES)]
bars = ax.bar(classes, weights, color=['#2ecc71','#f39c12','#e67e22','#e74c3c','#8e44ad'])
ax.set_title('Class Weights — Compensación por Desbalance', fontsize=12, fontweight='bold')
ax.set_xlabel('Clase'); ax.set_ylabel('Peso')
for bar, w in zip(bars, weights):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{w:.2f}', ha='center', fontweight='bold')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('/content/class_weights.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Class Weights configurados')

In [ ]:
# ──────────────────────────────────────────────
# 4.2 Configuración de Callbacks (Regularización del entrenamiento)
# ──────────────────────────────────────────────
import datetime

log_dir = '/content/logs/fit/' + datetime.datetime.now().strftime('%Y%m%d-%H%M%S')

callbacks = [
    # Detiene entrenamiento si val_loss no mejora en 7 epochs
    keras.callbacks.EarlyStopping(
        monitor              = 'val_loss',
        patience             = 7,
        restore_best_weights = True,
        verbose              = 1
    ),
    # Guarda el mejor modelo
    keras.callbacks.ModelCheckpoint(
        filepath         = '/content/best_model_phase1.h5',
        monitor          = 'val_accuracy',
        save_best_only   = True,
        verbose          = 1
    ),
    # Reduce learning rate si val_loss se estanca
    keras.callbacks.ReduceLROnPlateau(
        monitor   = 'val_loss',
        factor    = 0.3,
        patience  = 3,
        min_lr    = 1e-7,
        verbose   = 1
    ),
    # TensorBoard para monitoreo
    keras.callbacks.TensorBoard(
        log_dir          = log_dir,
        histogram_freq   = 1
    )
]

print('✅ Callbacks configurados:')
print('   • EarlyStopping       (patience=7)')
print('   • ModelCheckpoint     (guarda mejor val_accuracy)')
print('   • ReduceLROnPlateau   (factor=0.3, patience=3)')
print('   • TensorBoard         (logs en', log_dir + ')')

In [ ]:
# ──────────────────────────────────────────────
# 4.3 Resumen del Pipeline de Datos — Verificación Final
# ──────────────────────────────────────────────
print('=' * 55)
print('     RESUMEN DEL PIPELINE — ETAPAS COMPLETADAS')
print('=' * 55)
print(f'  Dataset           : APTOS 2019 (Kaggle)')
print(f'  Total imágenes    : {len(df_train)}')
print(f'  Train set         : {len(train_df)} ({len(train_df)/len(df_train)*100:.0f}%)')
print(f'  Val set           : {len(val_df)} ({len(val_df)/len(df_train)*100:.0f}%)')
print(f'  Tamaño imagen     : {IMG_SIZE}x{IMG_SIZE} px')
print(f'  Batch size        : {BATCH_SIZE}')
print(f'  Preprocesamiento  : Ben Graham (médico)')
print(f'  Augmentation      : Rotación 360°, Flip, Zoom, Brillo')
print(f'  Arquitectura      : EfficientNetB4 + Custom Head')
print(f'  Desbalance        : Class Weighting (balanced)')
print(f'  Parámetros modelo : {model.count_params():,}')
print('=' * 55)
print()
print('⏳ PENDIENTE:')
print('  [ ] 5. Modeling — Entrenamiento Fase 1 (feature extraction)')
print('  [ ]    Entrenamiento Fase 2 (fine-tuning capas superiores)')
print('  [ ] 6. Model Evaluation & Tuning')
print('      - Curvas de entrenamiento (loss & accuracy)')
print('      - Matriz de confusión')
print('      - Reporte de clasificación (precision, recall, F1)')
print('      - Curva ROC-AUC multiclase')
print('      - Grad-CAM (interpretabilidad)')
print('      - Hyperparameter tuning')
print('  [ ] 7. Deployment & Monitoring')
print('      - Exportar modelo (.h5 / TFLite)')
print('      - Demo de inferencia con nuevas imágenes')
print('      - Gradio/Streamlit demo (opcional)')

---
## ⏳ 5. MODELING — PENDIENTE
> Esta sección está pendiente de completar.

```python
# TODO: Fase 1 — Feature Extraction (base congelada)
# history_phase1 = model.fit(
#     train_generator,
#     validation_data  = val_generator,
#     epochs           = 20,
#     class_weight     = class_weights,
#     callbacks        = callbacks
# )

# TODO: Fase 2 — Fine Tuning (descongelar últimas capas)
# base_model.trainable = True
# fine_tune_at = 200
# for layer in base_model.layers[:fine_tune_at]:
#     layer.trainable = False
# model.compile(optimizer=keras.optimizers.Adam(1e-5), ...)
# history_phase2 = model.fit(...)
```

---
## ⏳ 6. MODEL EVALUATION & TUNING — PENDIENTE
> Esta sección está pendiente de completar.

```python
# TODO: Curvas de entrenamiento
# TODO: Matriz de confusión
# TODO: Classification report (precision, recall, F1)
# TODO: ROC-AUC multiclase
# TODO: Grad-CAM para interpretabilidad
# TODO: Hyperparameter tuning (Keras Tuner)
```

---
## ⏳ 7. DEPLOYMENT & MONITORING — PENDIENTE
> Esta sección está pendiente de completar.

```python
# TODO: Exportar modelo (.h5 y TFLite)
# TODO: Pipeline de inferencia para nuevas imágenes
# TODO: Demo interactivo con Gradio
```